In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
# from lightgbm.lgb import LGBMRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr

In [2]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []
        
        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -3.9)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -3.9)
            test_predictions_folds.append(predictions_test_fold)

        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)

        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df

In [3]:
#All fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/All_fingerprints_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/All_fingerprints_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 20188)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 20188)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.204589 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17668
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 4093
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.163774 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17848
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 4154
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threa

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2162,0.3412,0.4650,0.6528,0.8089,0.7782,0.2190,0.3420,0.4680,0.6555,0.8119,0.7970
DecisionTreeRegressor,0.2956,0.3804,0.5437,0.5252,0.7427,0.7282,0.2497,0.3493,0.4997,0.6073,0.7810,0.7782
RandomForestRegressor,0.2209,0.3449,0.4700,0.6452,0.8033,0.7759,0.2289,0.3442,0.4784,0.6400,0.8004,0.7875
GradientBoostingRegressor,0.2570,0.3760,0.5070,0.5872,0.7728,0.7387,0.2615,0.3758,0.5113,0.5887,0.7747,0.7549
AdaBoostRegressor,0.4553,0.5456,0.6748,0.2686,0.5982,0.5670,0.4252,0.5268,0.6521,0.3312,0.6462,0.6225
XGBRegressor,0.2110,0.3367,0.4593,0.6612,0.8137,0.7809,0.2124,0.3317,0.4609,0.6658,0.8162,0.7990
ExtraTreesRegressor,0.2212,0.3439,0.4703,0.6447,0.8033,0.7742,0.2324,0.3402,0.4821,0.6345,0.7970,0.7891
LinearRegression,0.6017,0.4866,0.7757,0.0336,0.5649,0.6429,0.4357,0.4432,0.6601,0.3147,0.6274,0.6730
KNeighborsRegressor,0.3163,0.4088,0.5624,0.4920,0.7136,0.6881,0.2963,0.3882,0.5444,0.5339,0.7365,0.7318
SVR,0.2810,0.3780,0.5301,0.5486,0.7435,0.7271,0.2831,0.3744,0.5321,0.5547,0.7477,0.7466


In [4]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.840063597113214, -6.903552652961309, -6.04...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.927528163323297, -6.241881086384121, -6.9...","[-7.018345836178957, -6.2780491460213605, -6.9...","[0.09101768237089003, 0.068242995778154, 0.192..."
1,DecisionTreeRegressor,"[-6.96, -6.96, -6.244999999999999, -5.05, -5.1...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.24, -6.85, -7.0, -6.244999999999999...","[-6.992, -6.328, -6.709999999999999, -6.784000...","[0.016000000000000014, 0.3639450507975071, 0.4..."
2,RandomForestRegressor,"[-6.6966, -6.8888000000000025, -6.172016666666...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.8603, -6.307933333333335, -6.819533333333...","[-6.898180000000001, -6.343433333333335, -6.79...","[0.044175893878901855, 0.08731774161074084, 0...."
3,GradientBoostingRegressor,"[-6.708410411828268, -6.781004739570915, -6.02...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.966378255884383, -6.217323220040958, -6.6...","[-7.010852745258626, -6.05214184759069, -6.693...","[0.18226915429861126, 0.18798172723978487, 0.1..."
4,AdaBoostRegressor,"[-6.00726475051561, -5.975086306098976, -5.745...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.093921568627448, -5.825461171398728, -6.0...","[-5.950513629746466, -5.721341131537711, -5.97...","[0.15107661563543082, 0.06708888816934401, 0.1..."
5,XGBRegressor,"[-6.6058135, -7.034152, -6.2442265, -5.1301036...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9824786, -6.2422824, -6.768956, -6.977634...","[-7.094104, -6.223756, -6.9432554, -6.804239, ...","[0.107741505, 0.090520084, 0.21957771, 0.24786..."
6,ExtraTreesRegressor,"[-6.789800000000001, -6.8241000000000005, -6.2...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.974000000000001, -6.512400000000002, -6.9...","[-6.941560000000001, -6.547650000000002, -6.87...","[0.040352526562781287, 0.09794170715277482, 0...."
7,LinearRegression,"[-4.058313657076598, -3.9, -5.428143197115932,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-9.484708483887491, -6.119877909874523, -10....","[-7.470073747261383, -4.386916965847902, -9.45...","[1.64826814366318, 0.8704615256080475, 0.67359..."
8,KNeighborsRegressor,"[-5.633333333333333, -6.113333333333333, -5.62...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -5.293333333333334, -5.786666666666666...","[-6.9093333333333335, -5.173333333333334, -6.0...","[0.04533333333333331, 0.29806039656418654, 0.3..."
9,SVR,"[-6.0727027819319, -6.480578944055306, -4.9772...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.657719427628311, -5.652057560383028, -5.9...","[-6.787475219635093, -5.646978141350594, -5.99...","[0.06986787211635924, 0.13849052940798776, 0.0..."


In [5]:
result_df.to_csv('Results/Fingerprints/Results_All_fingerprints_fp.csv')
prediction_df.to_csv('Results/Fingerprints/Prediction_data_All_fingerprints_fp.csv')

In [6]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]    
    df_cleaned = df.drop(columns=constant_columns)    
    return df_cleaned, constant_columns

In [7]:
#Low variance column removal
def remove_low_variance_columns(df, threshold=0.005):
    df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()    
    low_variance_columns = variances[variances < threshold].index.tolist()
    df_cleaned = df.drop(columns=low_variance_columns)   
    return df_cleaned, low_variance_columns

In [8]:
#All fingerprints constant removal
df_train = pd.read_csv('features/Fingerprints/Train/All_fingerprints_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/All_fingerprints_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 6820)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 6820)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.176095 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 17668
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 4093
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.202536 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17848
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 4154
[LightGBM] [Info] Start training from score 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2162,0.3412,0.4650,0.6528,0.8089,0.7782,0.2190,0.3420,0.4680,0.6555,0.8119,0.7970
DecisionTreeRegressor,0.2984,0.3808,0.5463,0.5207,0.7410,0.7231,0.2438,0.3484,0.4937,0.6166,0.7868,0.7785
RandomForestRegressor,0.2206,0.3446,0.4697,0.6457,0.8036,0.7758,0.2291,0.3443,0.4786,0.6396,0.8002,0.7877
GradientBoostingRegressor,0.2569,0.3759,0.5069,0.5874,0.7729,0.7392,0.2611,0.3757,0.5110,0.5893,0.7750,0.7551
AdaBoostRegressor,0.4482,0.5410,0.6695,0.2801,0.6053,0.5702,0.4233,0.5243,0.6506,0.3342,0.6461,0.6237
XGBRegressor,0.2110,0.3367,0.4593,0.6612,0.8137,0.7809,0.2124,0.3317,0.4609,0.6658,0.8162,0.7990
ExtraTreesRegressor,0.2199,0.3433,0.4689,0.6469,0.8046,0.7753,0.2320,0.3404,0.4817,0.6351,0.7974,0.7887
LinearRegression,0.6017,0.4866,0.7757,0.0336,0.5649,0.6429,0.4357,0.4432,0.6601,0.3147,0.6274,0.6731
KNeighborsRegressor,0.3163,0.4088,0.5624,0.4920,0.7136,0.6881,0.2963,0.3882,0.5444,0.5339,0.7365,0.7318
SVR,0.2810,0.3780,0.5301,0.5486,0.7435,0.7271,0.2831,0.3744,0.5321,0.5547,0.7477,0.7466


In [9]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.840063597113214, -6.903552652961309, -6.04...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.927528163323297, -6.241881086384121, -6.9...","[-7.018345836178957, -6.2780491460213605, -6.9...","[0.09101768237089003, 0.068242995778154, 0.192..."
1,DecisionTreeRegressor,"[-6.96, -6.96, -6.244999999999999, -5.05, -5.1...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.24, -7.0, -7.0, -6.27, -7.0, -6.85,...","[-7.0, -6.308, -6.748, -6.396000000000001, -5....","[0.0, 0.38232970065115257, 0.5039999999999999,..."
2,RandomForestRegressor,"[-6.6968, -6.8838133333333325, -6.16075, -5.33...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.880700000000002, -6.3494, -6.780033333333...","[-6.884680000000001, -6.3428200000000015, -6.7...","[0.049444287839951875, 0.10814574240348171, 0...."
3,GradientBoostingRegressor,"[-6.708410411828268, -6.781004739570915, -6.02...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.966378255884383, -6.217323220040959, -6.6...","[-7.020010743040404, -6.052141847590689, -6.65...","[0.1982933080840832, 0.18798172723978543, 0.12..."
4,AdaBoostRegressor,"[-6.093921568627449, -6.093921568627449, -6.09...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.108308115543351, -6.093921568627449, -6.0...","[-5.988832032980679, -5.814961732433493, -6.03...","[0.16626533645518793, 0.1431724068212968, 0.17..."
5,XGBRegressor,"[-6.6058135, -7.034152, -6.2442265, -5.1301036...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9824786, -6.2422824, -6.768956, -6.977634...","[-7.094104, -6.223756, -6.9432554, -6.804239, ...","[0.107741505, 0.090520084, 0.21957771, 0.24786..."
6,ExtraTreesRegressor,"[-6.752600000000001, -6.754000000000002, -6.10...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.959199999999999, -6.541400000000002, -6.8...","[-6.92368, -6.5402900000000015, -6.90537, -6.4...","[0.055278545566973744, 0.12832035068530778, 0...."
7,LinearRegression,"[-4.058313657077415, -3.9, -5.428143197115864,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-9.484708483885422, -6.119877909874816, -10....","[-7.470073747260626, -4.386916965847628, -9.45...","[1.6482681436629352, 0.8704615256082683, 0.673..."
8,KNeighborsRegressor,"[-5.633333333333333, -6.113333333333333, -5.62...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -5.293333333333334, -5.786666666666666...","[-6.9093333333333335, -5.173333333333334, -6.0...","[0.04533333333333331, 0.29806039656418654, 0.3..."
9,SVR,"[-6.072659844038206, -6.4806408917591245, -4.9...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.65779886745698, -5.651912177039307, -5.99...","[-6.787481944005177, -5.646873796183286, -5.99...","[0.06985410063700545, 0.13852338885188087, 0.0..."


In [10]:
X_train.columns

Index(['Morgan_fp_1', 'Morgan_fp_2', 'Morgan_fp_5', 'Morgan_fp_7',
       'Morgan_fp_11', 'Morgan_fp_12', 'Morgan_fp_13', 'Morgan_fp_14',
       'Morgan_fp_18', 'Morgan_fp_19',
       ...
       'SubFPC295', 'SubFPC296', 'SubFPC297', 'SubFPC298', 'SubFPC299',
       'SubFPC300', 'SubFPC301', 'SubFPC302', 'SubFPC303', 'SubFPC307'],
      dtype='object', length=6820)

In [11]:
result_df.to_csv('Results/Fingerprints/Results_All_const_rem_fingerprints.csv')
prediction_df.to_csv('Results/Fingerprints/Prediction_data_All_const_rem_fingerprints.csv')

In [12]:
#Morgan fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/morgan_fp_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/morgan_fp_test.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_morgan_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_morgan_fp

X_train shape:  (5568, 2048)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 2048)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019817 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1314
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 438
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019544 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1332
[LightGBM] [Info] Number of data points in the train set: 4454, number of use

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2632,0.3767,0.5130,0.5773,0.7610,0.7393,0.2676,0.3796,0.5173,0.5790,0.7630,0.7439
DecisionTreeRegressor,0.3473,0.4033,0.5893,0.4422,0.6959,0.6872,0.2756,0.3662,0.5249,0.5666,0.7573,0.7585
RandomForestRegressor,0.2482,0.3650,0.4982,0.6013,0.7756,0.7527,0.2531,0.3626,0.5030,0.6020,0.7761,0.7656
GradientBoostingRegressor,0.3089,0.4119,0.5558,0.5039,0.7160,0.6873,0.3172,0.4168,0.5632,0.5011,0.7153,0.6980
AdaBoostRegressor,0.5044,0.5706,0.7102,0.1899,0.5297,0.4962,0.4850,0.5575,0.6965,0.2371,0.5673,0.5513
XGBRegressor,0.2456,0.3633,0.4956,0.6056,0.7785,0.7518,0.2523,0.3624,0.5023,0.6031,0.7767,0.7647
ExtraTreesRegressor,0.3220,0.3956,0.5674,0.4829,0.7137,0.7008,0.2745,0.3651,0.5240,0.5682,0.7579,0.7615
LinearRegression,0.3877,0.4351,0.6227,0.3772,0.6472,0.6647,0.3519,0.4197,0.5932,0.4465,0.6770,0.6885
KNeighborsRegressor,0.3380,0.4221,0.5813,0.4572,0.6895,0.6691,0.3137,0.3962,0.5601,0.5066,0.7193,0.7221
SVR,0.3028,0.3939,0.5503,0.5136,0.7196,0.7009,0.2957,0.3849,0.5438,0.5349,0.7341,0.7271


In [13]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.04047151968858, -6.758768236948421, -5.010...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.871610886136016, -6.318404524451251, -6.9...","[-6.90787085364329, -6.009436232312133, -6.684...","[0.14896805501993032, 0.2085120060854812, 0.20..."
1,DecisionTreeRegressor,"[-4.92, -7.0, -6.244999999999999, -4.6, -6.244...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -5.89, -7.0, -6.0457692307692295, -5.3...","[-6.9319999999999995, -6.182, -6.9319999999999...","[0.13599999999999995, 0.4308781730373449, 0.13..."
2,RandomForestRegressor,"[-5.8146, -6.828700000000002, -5.7563533333333...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.7974, -6.310000000000003, -6.7697, -6.073...","[-6.759940000000002, -6.041770000000001, -6.72...","[0.08820826718624494, 0.20030371284072115, 0.0..."
3,GradientBoostingRegressor,"[-6.047878021312435, -6.601705847469999, -5.08...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.6984760178369855, -5.730917954324062, -6....","[-6.831197191736065, -5.641777376469747, -6.14...","[0.11172026085886101, 0.11435095062526847, 0.1..."
4,AdaBoostRegressor,"[-5.678769846297579, -5.678769846297579, -5.50...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.7487392499442045, -5.502801918662996, -5....","[-5.708296519139774, -5.497596700673268, -5.84...","[0.061885977046487434, 0.0070051096389411325, ..."
5,XGBRegressor,"[-5.694624, -6.9449267, -5.215449, -4.9689984,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.005003, -6.4993114, -6.9130077, -7.093382...","[-6.958537, -6.2593026, -6.946154, -6.6404176,...","[0.16194566, 0.24789083, 0.23921, 0.38723168, ..."
6,ExtraTreesRegressor,"[-4.944800000000005, -6.984399999999999, -6.04...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -5.88999999999999, -7.0, -6.1263863553...","[-6.94628, -6.058909999999995, -6.978239999999...","[0.10743999999999972, 0.4816452557640336, 0.04..."
7,LinearRegression,"[-6.539260685531751, -5.3913395837905735, -5.1...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-8.527993389985753, -5.631212621272983, -7.0...","[-8.691538396835558, -4.953759750635995, -6.32...","[1.8992721343071244, 0.4005647899105963, 1.735..."
8,KNeighborsRegressor,"[-5.633333333333333, -5.650000000000001, -5.64...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.3999999999999995, -5.38, -6.2266666666666...","[-6.789333333333333, -4.868666666666667, -6.20...","[0.1946666666666669, 0.27924978862023225, 0.24..."
9,SVR,"[-5.870446048710111, -6.018647297608707, -4.92...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.02258842118311, -5.5855302097521395, -5.9...","[-6.0881412266320805, -5.456713129391352, -5.9...","[0.05163928226318899, 0.07486121720878847, 0.0..."


In [14]:
df_morgan_fp.to_csv('Results/Fingerprints/Results_Morgan_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_data_Morgan_fp.csv')

In [16]:
#Morgan count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/count_morgan_fp_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/count_morgan_fp_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_morgan_count_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_morgan_count_fp

X_train shape:  (5568, 2048)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 2048)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 10.348686 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2049
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 447
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 10.953002 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2072
[LightGBM] [Info] Number of data points in the train set: 4454, number of u

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2364,0.3562,0.4862,0.6203,0.7888,0.7608,0.2374,0.3538,0.4872,0.6266,0.7940,0.7783
DecisionTreeRegressor,0.3010,0.3848,0.5486,0.5166,0.7377,0.7184,0.2571,0.3523,0.5070,0.5957,0.7738,0.7695
RandomForestRegressor,0.2310,0.3527,0.4806,0.6290,0.7931,0.7658,0.2407,0.3486,0.4906,0.6214,0.7886,0.7821
GradientBoostingRegressor,0.2797,0.3916,0.5289,0.5508,0.7491,0.7148,0.2859,0.3919,0.5347,0.5503,0.7508,0.7260
AdaBoostRegressor,0.5100,0.5756,0.7142,0.1808,0.5489,0.5350,0.4782,0.5581,0.6915,0.2478,0.5940,0.5806
XGBRegressor,0.2212,0.3439,0.4703,0.6447,0.8030,0.7743,0.2260,0.3422,0.4754,0.6445,0.8030,0.7874
ExtraTreesRegressor,0.2251,0.3478,0.4744,0.6385,0.7995,0.7718,0.2341,0.3436,0.4838,0.6318,0.7951,0.7863
LinearRegression,0.3454,0.4214,0.5877,0.4453,0.6752,0.6732,0.3411,0.4176,0.5840,0.4635,0.6843,0.7022
KNeighborsRegressor,0.3171,0.4110,0.5631,0.4908,0.7099,0.6860,0.2994,0.3904,0.5472,0.5290,0.7334,0.7327
SVR,0.2943,0.3851,0.5425,0.5273,0.7286,0.7136,0.2879,0.3777,0.5366,0.5471,0.7416,0.7383


In [17]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.4062427465985845, -6.545693258658244, -5.8...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.1474706956704726, -6.03588704486024, -6.5...","[-6.991103057669956, -6.037555429781047, -6.70...","[0.13543475250486692, 0.30389877056184306, 0.0..."
1,DecisionTreeRegressor,"[-5.36, -6.82, -5.15, -4.66, -4.28, -4.66, -4....",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.24, -7.0, -5.89, -6.27, -6.85, -6.8...","[-7.0, -5.946, -7.0, -6.123666666666667, -5.53...","[0.0, 0.5880000000000003, 0.0, 0.4952265474844..."
2,RandomForestRegressor,"[-6.020850000000001, -6.203100000000001, -6.05...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.7388999999999974, -5.859600000000004, -6....","[-6.83311, -5.813740000000001, -6.766658571428...","[0.07099363633453404, 0.3053586324307886, 0.04..."
3,GradientBoostingRegressor,"[-6.284638897070341, -6.303620105129672, -5.79...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.815868250959869, -5.710787449718063, -6.4...","[-6.968400070259442, -5.875540960306106, -6.38...","[0.08010448494976384, 0.14194338556562688, 0.0..."
4,AdaBoostRegressor,"[-5.59224924012159, -5.717515087719292, -5.539...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.651914414414416, -5.59224924012159, -5.75...","[-5.702332922477483, -5.576907653939182, -5.84...","[0.07330100148970659, 0.022137070903021742, 0...."
5,XGBRegressor,"[-6.1525593, -6.3216653, -6.2655725, -5.342836...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.401365, -6.179413, -7.374755, -6.995741, ...","[-7.2119675, -6.0614004, -7.2261353, -6.829355...","[0.15836193, 0.23354559, 0.23910433, 0.0967893..."
6,ExtraTreesRegressor,"[-6.089500000000002, -6.507600000000002, -6.12...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.901599999999999, -6.134800000000005, -6.6...","[-6.908119999999999, -6.142540000000004, -6.78...","[0.04077918096283923, 0.2634367749574852, 0.07..."
7,LinearRegression,"[-6.425810351122545, -5.195166788762071, -4.82...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.080293499556176, -5.200087349726744, -7.9...","[-7.037475431157465, -5.145110473007955, -7.64...","[0.6233713946540884, 0.18550315773520132, 0.38..."
8,KNeighborsRegressor,"[-5.633333333333333, -5.650000000000001, -5.62...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.3999999999999995, -5.38, -6.2266666666666...","[-6.789333333333333, -4.868666666666667, -6.20...","[0.1946666666666669, 0.27924978862023225, 0.24..."
9,SVR,"[-5.8835216110798765, -5.971567993147465, -5.1...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.018466220238241, -5.539984084273697, -5.9...","[-6.077576644603271, -5.420352676144331, -5.93...","[0.0485840342077941, 0.08873955035405413, 0.01..."


In [18]:
df_morgan_count_fp.to_csv('Results/Fingerprints/Results_Count_Morgan_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_data_Count_Morgan_fp.csv')

In [19]:
#AtomPairs2d fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/AtomPairs2D_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/AtomPairs2D_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_AtomPairs2D_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_AtomPairs2D_fp

X_train shape:  (5568, 780)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 780)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.678011 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 282
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 94
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wi

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4729,0.5231,0.6877,0.2404,0.4905,0.4049,0.4738,0.5280,0.6883,0.2547,0.5053,0.4390
DecisionTreeRegressor,0.4690,0.5171,0.6848,0.2468,0.4996,0.4173,0.4678,0.5205,0.6840,0.2642,0.5147,0.4671
RandomForestRegressor,0.4657,0.5163,0.6824,0.2520,0.5033,0.4169,0.4659,0.5209,0.6826,0.2672,0.5172,0.4625
GradientBoostingRegressor,0.4722,0.5237,0.6871,0.2416,0.4919,0.4079,0.4743,0.5279,0.6887,0.2540,0.5062,0.4523
AdaBoostRegressor,0.5512,0.5933,0.7425,0.1146,0.4132,0.3307,0.5438,0.5908,0.7374,0.1447,0.4443,0.3749
XGBRegressor,0.4682,0.5176,0.6843,0.2480,0.5003,0.4169,0.4667,0.5212,0.6832,0.2659,0.5161,0.4627
ExtraTreesRegressor,0.4687,0.5171,0.6846,0.2472,0.4999,0.4169,0.4680,0.5209,0.6841,0.2639,0.5143,0.4656
LinearRegression,0.4979,0.5380,0.7056,0.2003,0.4486,0.3824,0.4991,0.5404,0.7065,0.2149,0.4639,0.4199
KNeighborsRegressor,1.0399,0.7706,1.0198,-0.6703,0.1033,0.0413,1.0209,0.7623,1.0104,-0.6058,0.1123,0.0549
SVR,0.4989,0.5152,0.7063,0.1987,0.4681,0.3880,0.5037,0.5188,0.7097,0.2077,0.4798,0.4409


In [20]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-4.909314497886697, -6.111552803280551, -4.90...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.111552803280551, -4.909314497886697, -4.9...","[-6.210619938113033, -4.8974943325629745, -4.8...","[0.10995601512453465, 0.006133514899460441, 0...."
1,DecisionTreeRegressor,"[-4.906384803921563, -6.891818181818182, -4.90...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.891818181818182, -4.906384803921563, -4.9...","[-6.905297702297702, -4.892042722936078, -4.89...","[0.029975904951973498, 0.007554972370242409, 0..."
2,RandomForestRegressor,"[-4.904417636700295, -6.8975474147013145, -4.9...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.897547414701315, -4.904417636700295, -4.9...","[-6.907922054244575, -4.891981868002707, -4.89...","[0.027480858365760988, 0.0064928070601663525, ..."
3,GradientBoostingRegressor,"[-4.939840918219419, -6.41834612158003, -4.939...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.41834612158003, -4.939840918219419, -4.93...","[-6.447285198026674, -4.932545294644299, -4.93...","[0.07456595333968297, 0.0064940100699286685, 0..."
4,AdaBoostRegressor,"[-5.215743073047852, -5.263114324155363, -5.21...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.263114324155363, -5.215743073047852, -5.2...","[-5.673010994845329, -5.196889352041298, -5.19...","[0.2705239068997174, 0.014720120783385676, 0.0..."
5,XGBRegressor,"[-4.905683, -6.8888984, -4.905683, -4.905683, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.8888984, -4.905683, -4.905683, -5.710377,...","[-6.903943, -4.89107, -4.89107, -5.697889, -4....","[0.030578783, 0.0077610766, 0.0077610766, 0.23..."
6,ExtraTreesRegressor,"[-4.906384803921571, -6.8918181818181745, -4.9...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.8918181818181745, -4.906384803921571, -4....","[-6.905297702297702, -4.892042722936081, -4.89...","[0.029975904951973303, 0.007554972370245541, 0..."
7,LinearRegression,"[-4.980696835974732, -6.077738841685477, -4.98...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.077738841685477, -4.980696835974732, -4.9...","[-6.1226644205673875, -4.966436620305529, -4.9...","[0.10916274490237356, 0.007134747093014028, 0...."
8,KNeighborsRegressor,"[-7.0, -6.986666666666667, -7.0, -7.0, -7.0, -...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.986666666666667, -7.0, -7.0, -6.206666666...","[-6.984, -7.0, -7.0, -5.948666666666667, -7.0,...","[0.005333333333333456, 0.0, 0.0, 0.36911666328..."
9,SVR,"[-4.700178760220444, -6.900090345461479, -4.70...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.900090345461479, -4.700178760220444, -4.7...","[-6.900165170137579, -4.697933322332237, -4.69...","[0.00012757983806701038, 0.003880058887432729,..."


In [21]:
df_AtomPairs2D_fp.to_csv('Results/Fingerprints/Results_AtomPairs2D_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_data_AtomPairs2D_fp.csv')


In [22]:
#AtomPairs2d Count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/AtomPairs2DCount_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/AtomPairs2DCount_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_AtomPairs2DCount_fp , pred_df= train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_AtomPairs2DCount_fp

X_train shape:  (5568, 780)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 780)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.248814 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2865
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 133
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.217850 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2882
[LightGBM] [Info] Number of data points in the train set: 4454, number of used 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2381,0.3567,0.4880,0.6176,0.7875,0.7573,0.2460,0.3591,0.4960,0.6130,0.7848,0.7725
DecisionTreeRegressor,0.3185,0.3871,0.5644,0.4884,0.7205,0.7146,0.2644,0.3584,0.5142,0.5841,0.7684,0.7660
RandomForestRegressor,0.2328,0.3500,0.4825,0.6262,0.7913,0.7658,0.2446,0.3490,0.4946,0.6152,0.7848,0.7799
GradientBoostingRegressor,0.2803,0.3961,0.5294,0.5498,0.7494,0.7076,0.2926,0.3988,0.5409,0.5398,0.7429,0.7289
AdaBoostRegressor,0.5293,0.5951,0.7275,0.1499,0.5281,0.4627,0.4969,0.5787,0.7049,0.2184,0.5820,0.5031
XGBRegressor,0.2350,0.3506,0.4848,0.6225,0.7897,0.7650,0.2385,0.3447,0.4884,0.6248,0.7911,0.7822
ExtraTreesRegressor,0.2222,0.3460,0.4714,0.6431,0.8021,0.7745,0.2326,0.3433,0.4823,0.6341,0.7969,0.7845
LinearRegression,0.4162,0.4745,0.6451,0.3315,0.5797,0.6028,0.4038,0.4674,0.6355,0.3648,0.6048,0.6378
KNeighborsRegressor,0.2759,0.3805,0.5253,0.5568,0.7530,0.7202,0.2640,0.3679,0.5138,0.5848,0.7686,0.7544
SVR,0.3387,0.4128,0.5820,0.4559,0.6831,0.6738,0.3547,0.4196,0.5955,0.4421,0.6736,0.6806


In [23]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.769498897789005, -6.937127312828315, -6.81...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.034748839162398, -6.776804931522812, -6.8...","[-7.113290127572981, -6.755953336508185, -6.81...","[0.07927977034649857, 0.05502771164932075, 0.0..."
1,DecisionTreeRegressor,"[-6.51, -7.0, -7.0, -5.05, -7.0, -5.05, -5.05,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.11, -7.0, -7.0, -6.11, -6.8, -6.85,...","[-6.970000000000001, -6.67, -7.0, -6.778000000...","[0.060000000000000143, 0.40625115384451504, 0...."
2,RandomForestRegressor,"[-6.772900000000001, -6.9178, -6.8565666666666...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.933200000000002, -6.686400000000003, -6.9...","[-6.920140000000002, -6.722480000000002, -6.92...","[0.02578903643023542, 0.0904715071168812, 0.03..."
3,GradientBoostingRegressor,"[-6.681711585991144, -6.851244449978766, -6.55...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.111298630390628, -6.681711585991144, -6.8...","[-7.186987270473429, -6.660901688502126, -6.79...","[0.10042451912143581, 0.053977638252843775, 0...."
4,AdaBoostRegressor,"[-5.910666666666667, -5.68585553230739, -5.685...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.910666666666667, -6.0170761035988045, -5....","[-6.1914392496392505, -6.201047193062882, -6.1...","[0.1990673227468784, 0.20016443183663363, 0.19..."
5,XGBRegressor,"[-6.6733994, -6.9804363, -6.9170046, -5.244876...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0722523, -6.4999003, -7.087014, -6.596422...","[-7.0309434, -6.7597976, -7.022835, -6.633052,...","[0.08811137, 0.16679618, 0.049254216, 0.229982..."
6,ExtraTreesRegressor,"[-6.656700000000003, -6.884100000000001, -6.68...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.965100000000002, -6.554300000000004, -6.9...","[-6.953620000000003, -6.742740000000002, -6.95...","[0.015544696844905845, 0.11946049723653314, 0...."
7,LinearRegression,"[-5.732361429605066, -5.61441375434709, -5.315...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.604810056954641, -4.986989064300937, -5.7...","[-6.593928132253781, -4.978849312607559, -5.67...","[0.05551889105487907, 0.07421230040868694, 0.0..."
8,KNeighborsRegressor,"[-6.746666666666667, -7.0, -6.05, -4.96, -4.88...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.986666666666667, -6.746666666666667, -7.0...","[-6.989333333333333, -6.797333333333334, -7.0,...","[0.005333333333333102, 0.10133333333333318, 0...."
9,SVR,"[-7.097242596715828, -6.896524696353922, -6.00...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.02942118082642, -6.611920108828243, -6.67...","[-7.055863212992405, -6.581815153081419, -6.52...","[0.029133567672752787, 0.05562659746113107, 0...."


In [24]:
df_AtomPairs2DCount_fp.to_csv('Results/Fingerprints/Results_AtomPairs2D_Count_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_df_AtomPairs2D_Count_fp.csv')


In [25]:
#EState fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/EState_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/EState_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_estate_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_estate_fp

X_train shape:  (5568, 79)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 79)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.599595 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 45
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 15
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4987,0.5403,0.7062,0.1989,0.4462,0.3576,0.5139,0.5531,0.7169,0.1917,0.4387,0.3701
DecisionTreeRegressor,0.4807,0.5280,0.6933,0.2280,0.4809,0.3781,0.5116,0.5448,0.7153,0.1953,0.4505,0.3941
RandomForestRegressor,0.4761,0.5273,0.6900,0.2353,0.4861,0.3804,0.5088,0.5447,0.7133,0.1998,0.4534,0.3950
GradientBoostingRegressor,0.4828,0.5328,0.6949,0.2245,0.4738,0.3734,0.4867,0.5409,0.6976,0.2345,0.4848,0.3974
AdaBoostRegressor,0.6200,0.6330,0.7874,0.0041,0.3350,0.2659,0.6068,0.6307,0.7790,0.0456,0.3748,0.3796
XGBRegressor,0.4813,0.5281,0.6938,0.2269,0.4796,0.3802,0.5118,0.5451,0.7154,0.1950,0.4502,0.3953
ExtraTreesRegressor,0.4799,0.5277,0.6928,0.2292,0.4819,0.3794,0.5132,0.5452,0.7164,0.1927,0.4484,0.3942
LinearRegression,0.5020,0.5415,0.7085,0.1938,0.4406,0.3515,0.5091,0.5498,0.7135,0.1992,0.4469,0.3738
KNeighborsRegressor,1.0374,0.7533,1.0185,-0.6662,0.0607,-0.0063,1.0472,0.7575,1.0233,-0.6472,0.0224,-0.0434
SVR,0.5030,0.5202,0.7092,0.1921,0.4643,0.3740,0.5113,0.5301,0.7151,0.1958,0.4693,0.3893


In [26]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-4.725909084505892, -6.157864285599678, -5.20...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.157864285599678, -4.725909084505892, -5.2...","[-6.341078228842482, -4.785098423787131, -5.17...","[0.11092748975530736, 0.03002915439813857, 0.0..."
1,DecisionTreeRegressor,"[-4.677142857142856, -6.387333333333332, -5.13...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.387333333333332, -4.677142857142856, -5.1...","[-6.452906725146198, -4.750804141523363, -5.10...","[0.05644788211127162, 0.037324128282113664, 0...."
2,RandomForestRegressor,"[-4.682808658584028, -6.350122042329397, -5.12...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.350122042329398, -4.682808658584028, -5.1...","[-6.439855177418162, -4.752358232880523, -5.10...","[0.06421896488370232, 0.03583558693138541, 0.0..."
3,GradientBoostingRegressor,"[-4.716521135376747, -6.305582388490544, -5.18...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.305582388490544, -4.716521135376747, -5.1...","[-6.406281795487492, -4.803556606327592, -5.15...","[0.09003388798299372, 0.046653054674707484, 0...."
4,AdaBoostRegressor,"[-5.116787878787874, -5.7796319718593185, -5.7...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.7796319718593185, -5.116787878787874, -5....","[-5.974024682119785, -5.230241783763938, -5.83...","[0.10641242392632567, 0.10611131662800771, 0.0..."
5,XGBRegressor,"[-4.6779428, -6.385623, -5.136345, -5.136345, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.385623, -4.6779428, -5.136345, -5.0650125...","[-6.4517875, -4.7517877, -5.11008, -5.4007335,...","[0.056844383, 0.03744792, 0.016245445, 0.21314..."
6,ExtraTreesRegressor,"[-4.677142857142868, -6.387333333333341, -5.13...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.387333333333341, -4.677142857142868, -5.1...","[-6.452906725146202, -4.75080414152337, -5.109...","[0.05644788211127325, 0.03732412828211146, 0.0..."
7,LinearRegression,"[-4.732900901444904, -5.986727284967625, -5.18...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.986727284967625, -4.732900901444904, -5.1...","[-6.154225596429219, -4.781654916485074, -5.16...","[0.09165952883511957, 0.028247658582679025, 0...."
8,KNeighborsRegressor,"[-5.603333333333333, -6.986666666666667, -7.0,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.986666666666667, -5.603333333333333, -7.0...","[-6.984, -6.708666666666668, -7.0, -5.17866666...","[0.005333333333333456, 0.5531549913400807, 0.0..."
9,SVR,"[-4.58993325902177, -6.900096919543222, -4.840...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.900096919543222, -4.58993325902177, -4.84...","[-6.900054509281593, -4.613897089436945, -4.80...","[0.00018729940823278166, 0.016208042890321687,..."


In [27]:
df_estate_fp.to_csv('Results/Fingerprints/Results_EState_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_data_EState_fp.csv')


In [28]:
#Extended fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Extended_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Extended_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_extended_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_extended_fp

X_train shape:  (5568, 1024)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 1024)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 18.177090 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2274
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 758
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 17.051662 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2271
[LightGBM] [Info] Number of data points in the train set: 4454, number of u

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3707,0.4623,0.6089,0.4046,0.6371,0.5564,0.3773,0.4655,0.6143,0.4065,0.6396,0.5839
DecisionTreeRegressor,0.4343,0.4777,0.6590,0.3025,0.5861,0.5330,0.3869,0.4592,0.6220,0.3914,0.6318,0.5824
RandomForestRegressor,0.3629,0.4524,0.6024,0.4171,0.6473,0.5708,0.3688,0.4540,0.6073,0.4200,0.6486,0.5889
GradientBoostingRegressor,0.3909,0.4792,0.6252,0.3721,0.6123,0.5239,0.3999,0.4812,0.6323,0.3711,0.6119,0.5531
AdaBoostRegressor,0.6736,0.6657,0.8207,-0.0819,0.3589,0.3802,0.6490,0.6525,0.8056,-0.0208,0.3908,0.4333
XGBRegressor,0.3648,0.4528,0.6040,0.4141,0.6479,0.5683,0.3648,0.4523,0.6039,0.4263,0.6541,0.5935
ExtraTreesRegressor,0.4192,0.4735,0.6474,0.3268,0.5987,0.5389,0.3873,0.4603,0.6224,0.3907,0.6308,0.5804
LinearRegression,0.5203,0.5097,0.7213,0.1643,0.5169,0.5194,0.4345,0.4835,0.6591,0.3166,0.5811,0.5429
KNeighborsRegressor,0.4686,0.5081,0.6845,0.2474,0.5543,0.4730,0.4411,0.4928,0.6642,0.3061,0.5908,0.5311
SVR,0.4002,0.4642,0.6326,0.3572,0.6070,0.5447,0.4011,0.4623,0.6333,0.3691,0.6180,0.5691


In [29]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.341503394827989, -6.67958315034883, -5.702...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.211823516462028, -5.107529509712811, -5.2...","[-6.348669325189801, -5.352029476122501, -5.19...","[0.11538532491895247, 0.17117834521871747, 0.0..."
1,DecisionTreeRegressor,"[-5.62, -6.96, -6.6225, -4.7775, -6.6225, -4.2...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.24, -5.1, -7.0, -4.817499999999999,...","[-7.0, -6.0680000000000005, -5.414, -5.958, -4...","[0.0, 0.781547183476468, 0.8778519237320154, 0..."
2,RandomForestRegressor,"[-5.947, -6.961500000000002, -6.57966345238095...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.845482500000002, -5.3237040476190485, -5....","[-6.867873261904762, -5.760672476190478, -5.51...","[0.054344107573470496, 0.24145395483339124, 0...."
3,GradientBoostingRegressor,"[-5.1514653549749205, -6.695767752420868, -5.1...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.599842806663712, -5.006530569445064, -4.9...","[-6.618566390746916, -5.155840360937221, -4.88...","[0.09582437404837305, 0.1708739936982961, 0.02..."
4,AdaBoostRegressor,"[-5.512203032855949, -5.66736255572066, -5.632...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.639674863387984, -5.512203032855949, -5.5...","[-5.654386153179572, -5.5433998482942, -5.5651...","[0.04504461517235703, 0.08498211701624224, 0.0..."
5,XGBRegressor,"[-7.958757, -8.069417, -6.4450593, -4.728273, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.780029, -5.5089116, -5.316167, -6.0177774...","[-6.7899313, -5.817984, -5.2936177, -5.985423,...","[0.0748396, 0.24952494, 0.17611615, 0.2911796,..."
6,ExtraTreesRegressor,"[-5.620000000000004, -6.9600000000000035, -6.6...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9868000000000015, -6.240000000000006, -5....","[-6.9973600000000005, -6.085280000000003, -5.4...","[0.005279999999999419, 0.7339010162140408, 0.8..."
7,LinearRegression,"[-5.990408484125445, -6.820506325745262, -5.27...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.696422805234768, -4.5929259685105555, -4....","[-6.731583368885057, -4.510086203165137, -4.78...","[0.11509577774425042, 0.3374959461631407, 0.06..."
8,KNeighborsRegressor,"[-5.603333333333333, -6.986666666666667, -6.61...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.986666666666667, -4.513333333333333, -5.5...","[-6.952000000000001, -4.504, -5.33133333333333...","[0.05356615847093516, 0.029013406862651362, 0...."
9,SVR,"[-5.136290423157867, -6.817681894883761, -4.73...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.872411813970841, -4.883931687277617, -4.6...","[-6.8610577638864925, -4.923369067553784, -4.6...","[0.04013024861042333, 0.02510716174953571, 0.0..."


In [30]:
df_extended_fp.to_csv('Results/Fingerprints/Results_Extended_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_data_Extended_fp.csv')


In [31]:
#Fingerprinter fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Fingerprinter_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Fingerprinter_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_fingerprinter_fp , pred_df= train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_fingerprinter_fp

X_train shape:  (5568, 1024)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 1024)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 17.030083 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2259
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 753
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 17.407068 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2259
[LightGBM] [Info] Number of data points in the train set: 4454, number of u

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3824,0.4729,0.6184,0.3858,0.6213,0.5350,0.3875,0.4764,0.6225,0.3904,0.6255,0.5501
DecisionTreeRegressor,0.4410,0.4878,0.6641,0.2917,0.5743,0.5141,0.4016,0.4708,0.6337,0.3684,0.6137,0.5478
RandomForestRegressor,0.3781,0.4652,0.6149,0.3928,0.6283,0.5483,0.3836,0.4650,0.6193,0.3967,0.6311,0.5531
GradientBoostingRegressor,0.4003,0.4886,0.6327,0.3571,0.5995,0.5000,0.4024,0.4898,0.6344,0.3671,0.6091,0.5162
AdaBoostRegressor,0.6251,0.6422,0.7906,-0.0040,0.3785,0.3790,0.6150,0.6378,0.7842,0.0327,0.3969,0.4234
XGBRegressor,0.3786,0.4640,0.6153,0.3919,0.6304,0.5487,0.3763,0.4643,0.6134,0.4081,0.6400,0.5582
ExtraTreesRegressor,0.4280,0.4833,0.6542,0.3126,0.5846,0.5190,0.3983,0.4696,0.6311,0.3735,0.6171,0.5509
LinearRegression,0.5167,0.5144,0.7188,0.1701,0.5063,0.5030,0.4387,0.4950,0.6623,0.3100,0.5759,0.5428
KNeighborsRegressor,0.4928,0.5227,0.7020,0.2085,0.5166,0.4348,0.4652,0.5044,0.6820,0.2683,0.5544,0.5010
SVR,0.4070,0.4724,0.6380,0.3462,0.5976,0.5269,0.4047,0.4718,0.6362,0.3634,0.6134,0.5399


In [32]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.028656170722523, -6.436458993680707, -5.10...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.906587650047853, -4.521193533219602, -4.8...","[-6.054410871782312, -4.5799250786527494, -4.8...","[0.16034358518687514, 0.03634746457955012, 0.0..."
1,DecisionTreeRegressor,"[-7.0, -6.96, -5.195, -4.7775, -4.63, -4.26, -...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -4.77, -5.12, -5.11, -6.24, -5.195, -5...","[-7.0, -4.562666666666667, -5.006, -6.46599999...","[0.0, 0.10740163458305028, 0.22800000000000012..."
2,RandomForestRegressor,"[-5.867288888888892, -6.698379761904762, -5.17...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9621, -4.705171666666667, -5.030890238095...","[-6.929829404761906, -4.7051596493506525, -4.9...","[0.05406800474524195, 0.146014568142191, 0.138..."
3,GradientBoostingRegressor,"[-4.811053254680453, -6.560524530633439, -4.93...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.50159366691474, -4.691272909348114, -4.93...","[-6.3749964448330605, -4.643794370031959, -4.8...","[0.11498523870984269, 0.0670774988981201, 0.02..."
4,AdaBoostRegressor,"[-5.545141491395792, -5.711353399740146, -5.54...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.659300538047676, -5.545141491395792, -5.5...","[-5.666582437902202, -5.560338166808161, -5.56...","[0.03045448661777516, 0.07816076701707077, 0.0..."
5,XGBRegressor,"[-6.8988585, -6.772375, -5.14098, -4.735314, -...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9368434, -4.8132873, -4.814614, -5.271599...","[-6.8041, -4.7234664, -4.839012, -5.5932503, -...","[0.115867354, 0.1952058, 0.03391179, 0.2923629..."
6,ExtraTreesRegressor,"[-7.0, -6.982400000000002, -5.194999999999996,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -4.566199999999995, -5.120000000000004...","[-7.0, -4.548186666666668, -5.006000000000006,...","[0.0, 0.050421695506769226, 0.2279999999999983..."
7,LinearRegression,"[-5.965302835717242, -6.055186835700789, -5.24...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.062452919412785, -6.14723827259685, -4.96...","[-6.25970194162173, -5.159047282804899, -4.938...","[0.11892651491334028, 0.864569624404245, 0.076..."
8,KNeighborsRegressor,"[-5.603333333333333, -6.986666666666667, -6.49...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -5.2299999999999995, -5.52666666666666...","[-6.954666666666666, -5.053333333333333, -4.84...","[0.055521767503085344, 0.29412015685203635, 0...."
9,SVR,"[-4.97469508603877, -6.787865693658413, -4.749...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.834299846762822, -5.13393088691373, -4.68...","[-6.808609617884137, -5.140242981005375, -4.66...","[0.05436719664072799, 0.09581927686946433, 0.0..."


In [33]:
df_fingerprinter_fp.to_csv('Results/Fingerprints/Results_Fingerprinter_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_data_Fingerprinter_fp.csv')


In [34]:
#GraphOnly fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Graphonly_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Graphonly_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_graph_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_graph_fp

X_train shape:  (5568, 1024)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 1024)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 9.189949 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1272
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 424
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 11.025589 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1266
[LightGBM] [Info] Number of data points in the train set: 4454, number of us

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4048,0.4852,0.6362,0.3499,0.5915,0.5020,0.4220,0.4950,0.6496,0.3362,0.5801,0.5033
DecisionTreeRegressor,0.4241,0.4875,0.6512,0.3188,0.5778,0.4874,0.4318,0.4902,0.6571,0.3209,0.5720,0.5103
RandomForestRegressor,0.3983,0.4777,0.6311,0.3603,0.6019,0.5058,0.4172,0.4846,0.6459,0.3437,0.5876,0.5158
GradientBoostingRegressor,0.4191,0.4958,0.6474,0.3268,0.5736,0.4706,0.4311,0.5032,0.6566,0.3219,0.5700,0.4877
AdaBoostRegressor,0.5697,0.6132,0.7548,0.0850,0.4403,0.3864,0.5548,0.6074,0.7449,0.1273,0.4762,0.4306
XGBRegressor,0.4024,0.4793,0.6344,0.3537,0.5983,0.5015,0.4215,0.4875,0.6492,0.3371,0.5828,0.5135
ExtraTreesRegressor,0.4177,0.4855,0.6463,0.3291,0.5839,0.4901,0.4301,0.4895,0.6559,0.3234,0.5738,0.5121
LinearRegression,0.4727,0.5102,0.6875,0.2408,0.5176,0.4649,0.4605,0.5075,0.6786,0.2757,0.5356,0.4796
KNeighborsRegressor,0.5692,0.5685,0.7544,0.0858,0.4214,0.3199,0.5927,0.5710,0.7699,0.0677,0.3937,0.3166
SVR,0.4357,0.4845,0.6601,0.3002,0.5592,0.4734,0.4419,0.4860,0.6648,0.3049,0.5649,0.4834


In [35]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-4.96164085535268, -6.523507666650032, -5.211...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.491629404437911, -4.937017649619787, -5.2...","[-6.4289182559061455, -5.165378402031151, -5.1...","[0.10303669658925757, 0.14467005451213247, 0.0..."
1,DecisionTreeRegressor,"[-4.805, -7.0, -5.24828125, -4.773249999999999...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -4.805, -5.24828125, -4.82, -5.2482812...","[-7.0, -6.561, -5.220977855716688, -6.128, -5....","[0.0, 0.8780000000000002, 0.05149737679915188,..."
2,RandomForestRegressor,"[-5.1037246860084355, -6.956245238095238, -5.2...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -5.0161416817904305, -5.22810032538687...","[-6.986931483405485, -6.0056741063658565, -5.2...","[0.008912957480715298, 0.5035359852493747, 0.0..."
3,GradientBoostingRegressor,"[-5.038664960016449, -6.977291963948363, -4.99...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.586604756763464, -5.038664960016449, -4.9...","[-6.4216259234566095, -5.258571392778277, -4.9...","[0.116171309076214, 0.25409650952006296, 0.029..."
4,AdaBoostRegressor,"[-5.36726923076923, -5.732677165554014, -5.482...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.670302491103192, -5.36726923076923, -5.48...","[-5.647818765823838, -5.27287269463054, -5.369...","[0.052596898999356725, 0.17320233293038295, 0...."
5,XGBRegressor,"[-5.5786085, -6.9957724, -5.2041016, -4.785585...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0040655, -5.3975577, -5.2041016, -5.47094...","[-7.032304, -6.1480575, -5.1658397, -5.7655864...","[0.016728042, 0.40852296, 0.0478423, 0.2382122..."
6,ExtraTreesRegressor,"[-4.805000000000006, -7.0, -5.2482812499999945...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -4.805000000000006, -5.248281249999994...","[-7.0, -6.561000000000002, -5.220977855716685,...","[0.0, 0.8779999999999976, 0.051497376799150944..."
7,LinearRegression,"[-5.399204355186469, -7.173360028618376, -5.14...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.725847484397764, -5.142470858330996, -5.1...","[-6.751835244659221, -4.958592152976298, -5.11...","[0.05838385654952785, 0.13927347557359543, 0.0..."
8,KNeighborsRegressor,"[-5.1933333333333325, -6.986666666666667, -7.0...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -4.420000000000001, -7.0, -5.383333333...","[-7.0, -4.689333333333334, -7.0, -5.4773333333...","[0.0, 0.3536313208853409, 0.0, 0.1582459407939..."
9,SVR,"[-4.796560296375508, -6.9946070980248845, -4.7...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.900397950382591, -4.6914119002670205, -4....","[-6.889560766396537, -4.701023503031583, -4.72...","[0.02084383244542893, 0.03903278309047845, 0.0..."


In [36]:
df_graph_fp.to_csv('Results/Fingerprints/Results_Graphonly_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_data_Graphonly_fp.csv')


In [37]:
#KlekotaRoth fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/KlekotaRoth_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/KlekotaRoth_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_KlekotaRoth_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_KlekotaRoth_fp

X_train shape:  (5568, 4860)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 4860)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 6.588532 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 840
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 280
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 7.607137 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 888
[LightGBM] [Info] Number of data points in the train set: 4454, number of used 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2948,0.4067,0.5430,0.5265,0.7269,0.6893,0.3068,0.4125,0.5539,0.5174,0.7206,0.6892
DecisionTreeRegressor,0.3749,0.4286,0.6123,0.3979,0.6625,0.6448,0.3303,0.4114,0.5747,0.4805,0.7023,0.6864
RandomForestRegressor,0.2844,0.3963,0.5333,0.5432,0.7371,0.7038,0.2963,0.3982,0.5443,0.5340,0.7312,0.7074
GradientBoostingRegressor,0.3356,0.4399,0.5793,0.4609,0.6839,0.6370,0.3448,0.4424,0.5872,0.4577,0.6816,0.6563
AdaBoostRegressor,0.5254,0.5847,0.7248,0.1562,0.5002,0.4404,0.5090,0.5770,0.7135,0.1994,0.5357,0.4841
XGBRegressor,0.2746,0.3899,0.5240,0.5589,0.7480,0.7065,0.2915,0.3979,0.5399,0.5415,0.7364,0.7087
ExtraTreesRegressor,0.3604,0.4242,0.6003,0.4212,0.6730,0.6492,0.3260,0.4094,0.5710,0.4872,0.7058,0.6895
LinearRegression,0.3853,0.4569,0.6207,0.3812,0.6264,0.6150,0.3824,0.4535,0.6184,0.3985,0.6355,0.6335
KNeighborsRegressor,0.3767,0.4538,0.6138,0.3949,0.6434,0.5926,0.3884,0.4515,0.6232,0.3891,0.6387,0.6151
SVR,0.3201,0.4135,0.5658,0.4859,0.7014,0.6682,0.3243,0.4120,0.5695,0.4899,0.7045,0.6827


In [38]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.125041915090161, -6.117677414554641, -4.93...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.560304036611419, -4.831724684821965, -5.6...","[-6.751864433416634, -5.003920967002328, -5.46...","[0.16970117548376093, 0.1712581557145138, 0.11..."
1,DecisionTreeRegressor,"[-4.77, -7.0, -4.47, -4.39, -4.515000000000001...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -4.42, -4.32, -5.89, -6.24, -4.39, -5....","[-6.83, -5.287999999999999, -5.09, -5.88733333...","[0.10751744044572485, 0.5398388648476505, 0.96..."
2,RandomForestRegressor,"[-5.262516904761903, -6.627516666666663, -4.99...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.605766666666665, -4.917626666666666, -5.1...","[-6.757345666666666, -5.051375333333334, -5.22...","[0.07946928683879577, 0.19743027346213496, 0.0..."
3,GradientBoostingRegressor,"[-4.992119371038237, -6.427306283642587, -4.95...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.577211343856722, -4.845939227714817, -5.1...","[-6.663899521582624, -4.874287551214881, -5.07...","[0.08545059744037325, 0.05710901931843326, 0.0..."
4,AdaBoostRegressor,"[-5.556448087431711, -5.717351768809163, -5.47...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.717351768809163, -5.476622276029057, -5.6...","[-5.70848126598212, -5.442484648208121, -5.628...","[0.0282270747356428, 0.04265128366799348, 0.07..."
5,XGBRegressor,"[-4.828297, -6.877285, -5.3506255, -4.892547, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.1125884, -4.818136, -5.307983, -6.706802,...","[-7.0918207, -5.0689874, -5.176355, -5.94381, ...","[0.09834208, 0.31822863, 0.16216548, 0.5459288..."
6,ExtraTreesRegressor,"[-4.751699999999994, -7.0, -4.5001666666666775...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -4.4006500000000015, -4.31999999999999...","[-6.830000000000004, -5.1742099999999995, -5.0...","[0.10751744044572457, 0.49740787327906105, 0.9..."
7,LinearRegression,"[-4.984573361495496, -6.342493798563343, -4.71...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.71298171747597, -4.924586623426304, -4.93...","[-6.731726174032529, -4.980760484931328, -4.90...","[0.06306940746010592, 0.050664582908140944, 0...."
8,KNeighborsRegressor,"[-4.8999999999999995, -7.0, -5.626666666666666...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.986666666666667, -5.1933333333333325, -5....","[-6.904000000000001, -5.116, -5.52, -5.6146666...","[0.04165466493816895, 0.3840856385987085, 0.33..."
9,SVR,"[-4.8563284942155285, -6.853505023229131, -4.6...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9084423966175805, -4.7208613522662, -4.81...","[-6.895413500766639, -4.768705892617846, -4.78...","[0.02556027153464319, 0.02845576899169343, 0.0..."


In [39]:
df_KlekotaRoth_fp.to_csv('Results/Fingerprints/Results_KlekotaRoth_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_data_KlekotaRoth_fp.csv')


In [40]:
#KlekotaRoth Count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/KlekotaRothCount_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/KlekotaRothCount_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_KlekotaRothCount_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_KlekotaRothCount_fp

X_train shape:  (5568, 4860)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 4860)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 5.528803 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2839
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 339
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 6.718182 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2886
[LightGBM] [Info] Number of data points in the train set: 4454, number of use

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2423,0.3609,0.4922,0.6109,0.7831,0.7549,0.2572,0.3702,0.5071,0.5955,0.7727,0.7524
DecisionTreeRegressor,0.3034,0.3836,0.5508,0.5127,0.7336,0.7185,0.2751,0.3655,0.5245,0.5672,0.7577,0.7512
RandomForestRegressor,0.2308,0.3531,0.4805,0.6292,0.7933,0.7668,0.2510,0.3573,0.5009,0.6053,0.7781,0.7655
GradientBoostingRegressor,0.2933,0.4028,0.5416,0.5289,0.7339,0.6986,0.3002,0.4037,0.5479,0.5277,0.7310,0.7167
AdaBoostRegressor,0.4892,0.5620,0.6994,0.2142,0.5668,0.5422,0.4678,0.5483,0.6840,0.2642,0.6030,0.5868
XGBRegressor,0.2244,0.3458,0.4737,0.6396,0.7998,0.7686,0.2404,0.3513,0.4903,0.6219,0.7886,0.7685
ExtraTreesRegressor,0.2261,0.3492,0.4755,0.6368,0.7984,0.7705,0.2363,0.3485,0.4861,0.6283,0.7930,0.7777
LinearRegression,0.3767,0.4368,0.6138,0.3949,0.6390,0.6514,0.3641,0.4346,0.6034,0.4274,0.6569,0.6792
KNeighborsRegressor,0.3146,0.4082,0.5609,0.4947,0.7149,0.6768,0.3125,0.4001,0.5590,0.5084,0.7224,0.7063
SVR,0.2738,0.3721,0.5232,0.5603,0.7521,0.7292,0.2839,0.3781,0.5329,0.5534,0.7463,0.7324


In [41]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.492646756819686, -6.671203448100149, -6.10...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.017822395221834, -6.4832261761994, -6.734...","[-7.1393326663528685, -6.500053973461055, -6.6...","[0.07121953546486649, 0.1041300514133717, 0.16..."
1,DecisionTreeRegressor,"[-6.24, -7.0, -7.0, -5.92, -5.15, -4.77, -4.66...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.24, -7.0, -7.0, -7.0, -5.74, -6.85,...","[-7.0, -6.544, -6.714, -6.396000000000001, -6....","[0.0, 0.37232244090304295, 0.5719999999999998,..."
2,RandomForestRegressor,"[-6.753299999999999, -6.7435, -5.8918183333333...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9747, -6.675100000000002, -6.711900000000...","[-6.956280000000001, -6.666660000000002, -6.57...","[0.020655788534936246, 0.22052824399609103, 0...."
3,GradientBoostingRegressor,"[-6.306765728960991, -6.740890111508338, -5.90...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.966656487946471, -6.509568526317411, -6.7...","[-7.052052948021445, -6.412916556849529, -6.57...","[0.08935355161163736, 0.08647878769401872, 0.1..."
4,AdaBoostRegressor,"[-5.888639218475278, -6.058695652173912, -5.57...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.978421665373558, -5.950096361543068, -5.9...","[-6.0447204779027945, -6.000479984307203, -6.0...","[0.09280819984922767, 0.0643901642547337, 0.11..."
5,XGBRegressor,"[-6.939283, -7.0826044, -6.328767, -5.494405, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.07413, -6.4656925, -7.2120156, -6.8006625...","[-7.1830025, -6.5939116, -7.1804495, -6.522080...","[0.10010202, 0.21272537, 0.1375032, 0.5282287,..."
6,ExtraTreesRegressor,"[-6.7742, -6.856700000000001, -6.0806000000000...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9967999999999995, -6.670500000000003, -6....","[-6.99148, -6.727700000000003, -6.754480000000...","[0.007706464818578666, 0.1885742612341338, 0.0..."
7,LinearRegression,"[-5.050148713750407, -6.485095815192805, -4.80...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.982710324332498, -5.243664960563693, -5.1...","[-6.117032465766295, -5.297888300945132, -5.12...","[0.10638329220629432, 0.06832968975180435, 0.0..."
8,KNeighborsRegressor,"[-4.8999999999999995, -7.0, -5.62, -5.55, -4.6...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.986666666666667, -5.1933333333333325, -6....","[-6.989333333333333, -5.575999999999999, -6.40...","[0.005333333333333102, 0.21866666666666723, 0...."
9,SVR,"[-6.210611557350038, -6.810520251880828, -5.29...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.324703775739054, -6.160695200866523, -6.7...","[-7.252705019577194, -6.313685929988953, -6.69...","[0.07783128759943043, 0.09850631938078686, 0.0..."


In [42]:
df_KlekotaRothCount_fp.to_csv('Results/Fingerprints/Results_KlekotaRoth_Count_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_data_KlekotaRoth_Count_fp.csv')


In [43]:
#MACCS fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/MACCS_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/MACCS_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_MACCS_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_MACCS_fp

X_train shape:  (5568, 166)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 166)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.892232 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 240
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 80
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.922875 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 243
[LightGBM] [Info] Number of data points in the train set: 4454, number of used fea

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3568,0.4417,0.5973,0.4269,0.6537,0.6285,0.3611,0.4478,0.6009,0.4320,0.6583,0.6371
DecisionTreeRegressor,0.3947,0.4486,0.6282,0.3661,0.6245,0.6136,0.3592,0.4344,0.5993,0.4351,0.6631,0.6483
RandomForestRegressor,0.3544,0.4354,0.5953,0.4307,0.6589,0.6358,0.3441,0.4301,0.5866,0.4588,0.6781,0.6569
GradientBoostingRegressor,0.3820,0.4639,0.6180,0.3865,0.6259,0.5949,0.3872,0.4654,0.6223,0.3909,0.6304,0.6056
AdaBoostRegressor,0.5644,0.6065,0.7512,0.0936,0.4421,0.4254,0.5411,0.5973,0.7356,0.1489,0.4902,0.4629
XGBRegressor,0.3523,0.4336,0.5935,0.4342,0.6620,0.6375,0.3493,0.4340,0.5911,0.4505,0.6721,0.6619
ExtraTreesRegressor,0.3841,0.4453,0.6198,0.3831,0.6342,0.6191,0.3546,0.4331,0.5955,0.4422,0.6681,0.6525
LinearRegression,0.4325,0.4895,0.6577,0.3053,0.5547,0.5510,0.4358,0.4916,0.6602,0.3145,0.5622,0.5615
KNeighborsRegressor,0.5588,0.5360,0.7475,0.1025,0.4515,0.4216,0.5622,0.5317,0.7498,0.1158,0.4550,0.4229
SVR,0.3790,0.4353,0.6156,0.3913,0.6330,0.6164,0.3906,0.4378,0.6250,0.3856,0.6289,0.6276


In [44]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-4.857661314810258, -6.1545726846430435, -4.7...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.1545726846430435, -4.789776956735031, -5....","[-6.324776477076417, -4.923709994228924, -5.12...","[0.1471621189617364, 0.12384822213803978, 0.05..."
1,DecisionTreeRegressor,"[-4.8999999999999995, -7.0, -4.801176470588235...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -5.816666666666666, -5.283818181818181...","[-6.987333333333334, -5.214666666666666, -5.19...","[0.025333333333333388, 0.389788261609927, 0.07..."
2,RandomForestRegressor,"[-4.932299642857139, -6.969599365079368, -4.78...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.8928603518562435, -5.795551746031743, -5....","[-6.9210414878426665, -5.214529498556998, -5.1...","[0.02004260080647954, 0.3840743514554297, 0.07..."
3,GradientBoostingRegressor,"[-4.6994403181483815, -6.307201951082792, -4.9...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.285734143404252, -4.6994403181483815, -4....","[-6.459950560931001, -4.773061981906315, -4.96...","[0.10585490721641294, 0.04375089904337202, 0.0..."
4,AdaBoostRegressor,"[-5.449511742892457, -5.842415413533832, -5.36...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.630889169827214, -5.449511742892457, -5.4...","[-5.671434217211305, -5.4521721071343014, -5.4...","[0.07872007928703734, 0.13413461352123263, 0.1..."
5,XGBRegressor,"[-5.0321903, -6.953666, -4.7108836, -4.8500757...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.769061, -5.2195206, -5.235374, -6.030132,...","[-6.86866, -5.099058, -5.158365, -6.0867405, -...","[0.053263526, 0.15021178, 0.07277264, 0.404790..."
6,ExtraTreesRegressor,"[-4.8999999999999915, -7.0, -4.801176470588242...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -5.81666666666667, -5.283818181818176,...","[-6.987333333333334, -5.214666666666666, -5.19...","[0.025333333333331966, 0.3897882616099315, 0.0..."
7,LinearRegression,"[-4.776537049766552, -6.140061542564499, -5.04...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.066221140160493, -4.784744680995197, -5.0...","[-6.252314501889851, -4.8253032058161, -4.9767...","[0.10058163401112154, 0.0229361972391032, 0.03..."
8,KNeighborsRegressor,"[-4.8999999999999995, -6.986666666666667, -5.8...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.986666666666667, -5.816666666666666, -7.0...","[-6.989333333333333, -5.58, -6.957333333333334...","[0.005333333333333102, 0.5033929324538081, 0.0..."
9,SVR,"[-4.70568369930923, -6.900290191507951, -4.569...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.900329591151843, -4.6301704760780815, -4....","[-6.893450044267373, -4.690032275712416, -4.72...","[0.013673245320362728, 0.045150082168252274, 0..."


In [45]:
df_MACCS_fp.to_csv('Results/Fingerprints/Results_MACCS_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_data_MACCS_fp.csv')


In [46]:
#PubChem fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/PubChem_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/PubChem_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_PubChem_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_PubChem_fp

X_train shape:  (5568, 881)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 881)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 5.224488 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 786
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 262
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 5.803928 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 801
[LightGBM] [Info] Number of data points in the train set: 4454, number of used fe

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3773,0.4709,0.6142,0.3941,0.6279,0.5369,0.3782,0.4698,0.6149,0.4052,0.6373,0.5604
DecisionTreeRegressor,0.4126,0.4782,0.6423,0.3374,0.5977,0.5239,0.3997,0.4644,0.6322,0.3713,0.6165,0.5611
RandomForestRegressor,0.3722,0.4650,0.6101,0.4022,0.6355,0.5432,0.3740,0.4590,0.6116,0.4117,0.6424,0.5683
GradientBoostingRegressor,0.3852,0.4791,0.6207,0.3813,0.6199,0.5158,0.3908,0.4812,0.6252,0.3852,0.6234,0.5430
AdaBoostRegressor,0.6497,0.6543,0.8060,-0.0435,0.3852,0.3822,0.6341,0.6454,0.7963,0.0026,0.4040,0.4207
XGBRegressor,0.3784,0.4670,0.6152,0.3922,0.6304,0.5362,0.3676,0.4555,0.6063,0.4218,0.6504,0.5770
ExtraTreesRegressor,0.4046,0.4751,0.6361,0.3501,0.6057,0.5288,0.3952,0.4632,0.6287,0.3784,0.6212,0.5619
LinearRegression,0.4196,0.4891,0.6478,0.3261,0.5767,0.5091,0.4102,0.4906,0.6405,0.3547,0.5978,0.5385
KNeighborsRegressor,0.5698,0.5695,0.7549,0.0848,0.4750,0.3932,0.5594,0.5608,0.7479,0.1201,0.4943,0.4410
SVR,0.3970,0.4677,0.6301,0.3623,0.6115,0.5232,0.4044,0.4663,0.6359,0.3639,0.6140,0.5522


In [47]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.1994899216778245, -6.240788244804692, -5.5...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.927121938752764, -6.197447348410936, -6.2...","[-7.073953209196967, -5.89093389024792, -6.057...","[0.28388200545792014, 0.3376684210655568, 0.25..."
1,DecisionTreeRegressor,"[-6.89, -7.0, -5.511875000000001, -4.979759036...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.873333333333334, -6.89, -7.0, -6.87...","[-6.948, -6.974666666666667, -6.88666666666666...","[0.049558046773455613, 0.05066666666666641, 0...."
2,RandomForestRegressor,"[-6.818967306707621, -6.983099999999999, -5.50...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.955114285714284, -6.841488888888886, -6.8...","[-6.9555808571428575, -6.860906972582972, -6.8...","[0.009528023783791072, 0.03249238141094426, 0...."
3,GradientBoostingRegressor,"[-6.907071585862282, -7.172450844770587, -5.31...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.74666000703713, -6.655557692980505, -6.28...","[-7.707740324914442, -6.387403893775804, -6.33...","[0.04732565494835182, 0.25105835136057797, 0.0..."
4,AdaBoostRegressor,"[-5.749535312197648, -5.810665456745329, -5.26...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.749535312197648, -5.711822206664993, -5.7...","[-5.829946991573004, -5.746845029106277, -5.77...","[0.07006843441910539, 0.09770139101207222, 0.0..."
5,XGBRegressor,"[-8.033032, -8.281252, -5.5146046, -4.9951444,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.4196534, -8.069413, -7.022901, -6.6698036...","[-7.2394004, -7.2627625, -6.880005, -6.413217,...","[0.14554702, 0.40653086, 0.105730854, 0.244854..."
6,ExtraTreesRegressor,"[-6.88999999999999, -7.0, -5.511874999999992, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.873333333333334, -6.88999999999999,...","[-6.947999999999996, -6.962253333333334, -6.89...","[0.04955804677346036, 0.05054238089629963, 0.0..."
7,LinearRegression,"[-6.454378806361565, -7.042834880222975, -5.54...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.754821979066369, -6.1170923739726195, -6....","[-7.906389088392058, -6.153844334242668, -6.73...","[0.10274360794520739, 0.06936199191737878, 0.0..."
8,KNeighborsRegressor,"[-6.963333333333334, -7.0, -6.836666666666666,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.973333333333334, -7.0, -6.963333333333334...","[-6.978666666666667, -7.0, -6.970666666666668,...","[0.006531972647421959, 0.0, 0.0146666666666664..."
9,SVR,"[-6.34629357315515, -6.995601509379079, -5.250...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9309357053079195, -6.326592783684195, -6....","[-6.924960323371172, -6.501746271315438, -6.51...","[0.029356663228314532, 0.10481712022016562, 0...."


In [48]:
df_PubChem_fp.to_csv('Results/Fingerprints/Results_PubChem_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_data_PubChem_fp.csv')


In [49]:
#Substructure fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Substructure_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Substructure_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_Substructure_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_Substructure_fp

X_train shape:  (5568, 307)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 307)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.244889 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 90
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 30
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4713,0.5262,0.6865,0.2430,0.4930,0.3849,0.4779,0.5288,0.6913,0.2482,0.4984,0.4093
DecisionTreeRegressor,0.4584,0.5139,0.6771,0.2637,0.5166,0.3952,0.4765,0.5185,0.6903,0.2505,0.5049,0.4355
RandomForestRegressor,0.4521,0.5125,0.6724,0.2739,0.5243,0.3962,0.4728,0.5189,0.6876,0.2564,0.5088,0.4305
GradientBoostingRegressor,0.4604,0.5202,0.6785,0.2606,0.5107,0.3945,0.4782,0.5284,0.6915,0.2478,0.4980,0.4094
AdaBoostRegressor,0.5566,0.6016,0.7461,0.1060,0.4208,0.3082,0.5535,0.6023,0.7440,0.1294,0.4398,0.3800
XGBRegressor,0.4529,0.5124,0.6730,0.2725,0.5241,0.3973,0.4745,0.5187,0.6889,0.2536,0.5070,0.4224
ExtraTreesRegressor,0.4560,0.5138,0.6753,0.2676,0.5199,0.3949,0.4771,0.5189,0.6907,0.2496,0.5043,0.4359
LinearRegression,0.4777,0.5292,0.6912,0.2327,0.4836,0.3820,0.4836,0.5324,0.6954,0.2394,0.4895,0.4023
KNeighborsRegressor,0.7088,0.6313,0.8419,-0.1384,0.2374,0.1498,0.7122,0.6330,0.8439,-0.1202,0.2182,0.1711
SVR,0.4761,0.5092,0.6900,0.2353,0.5022,0.3991,0.4914,0.5128,0.7010,0.2271,0.4961,0.4214


In [50]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.172100199575506, -6.120709531257888, -5.18...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.120709531257888, -5.172100199575506, -5.1...","[-6.22398746566014, -5.233334591061803, -5.172...","[0.0686907417759841, 0.03972171275612506, 0.01..."
1,DecisionTreeRegressor,"[-5.088333333333333, -6.900833333333334, -5.16...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.900833333333334, -5.088333333333333, -5.1...","[-6.910515384615384, -5.566761904761904, -5.13...","[0.029148345781560968, 0.2967080380844876, 0.0..."
2,RandomForestRegressor,"[-5.076318534492043, -6.9064871039392335, -5.1...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.906487103939233, -5.076318534492043, -5.1...","[-6.912931768470837, -5.570280502880205, -5.13...","[0.027500319917382752, 0.3097371555306215, 0.0..."
3,GradientBoostingRegressor,"[-5.024930619183727, -6.661899315387622, -5.13...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.661899315387622, -5.024930619183727, -5.1...","[-6.536908761297781, -5.096306632740621, -5.13...","[0.08174153875254485, 0.04785955931702074, 0.0..."
4,AdaBoostRegressor,"[-5.342141327623121, -5.63159090909091, -5.342...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.63159090909091, -5.342141327623121, -5.34...","[-5.637543182821728, -5.367585257610203, -5.36...","[0.169298852286895, 0.061535232829034184, 0.06..."
5,XGBRegressor,"[-5.1035223, -6.885591, -5.159991, -5.159991, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.885591, -5.1035223, -5.159991, -4.8634834...","[-6.900486, -5.54208, -5.136771, -4.933072, -7...","[0.03105432, 0.27579597, 0.015623674, 0.165881..."
6,ExtraTreesRegressor,"[-5.088333333333323, -6.900833333333335, -5.16...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.900833333333335, -5.088333333333323, -5.1...","[-6.9105153846153895, -5.566761904761902, -5.1...","[0.02914834578156092, 0.2967080380844896, 0.01..."
7,LinearRegression,"[-5.065006907604286, -6.008936960130976, -5.15...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.008936960130976, -5.065006907604286, -5.1...","[-6.173079197940906, -5.048811315752249, -5.13...","[0.08923935779530826, 0.03235647028391439, 0.0..."
8,KNeighborsRegressor,"[-5.593333333333334, -6.986666666666667, -7.0,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.986666666666667, -5.593333333333334, -7.0...","[-6.984, -6.621333333333334, -7.0, -5.12133333...","[0.005333333333333456, 0.5474693294301212, 0.0..."
9,SVR,"[-4.676356862063068, -6.89993921309047, -4.839...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.89993921309047, -4.676356862063068, -4.83...","[-6.9000804193900676, -4.794578638997756, -4.8...","[0.00010194231556079242, 0.07319760882122567, ..."


In [51]:
df_Substructure_fp.to_csv('Results/Fingerprints/Results_Substructure_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_data_Substructure_fp.csv')


In [52]:
#Substructure Count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/SubstructureCount_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/SubstructureCount_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_SubstructureCount_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_SubstructureCount_fp

X_train shape:  (5568, 307)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 307)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.165183 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 513
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 40
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.763366 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 4454, number of used fea

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


0.4625507486377356


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2643,0.3791,0.5141,0.5755,0.7605,0.7224,0.2650,0.3740,0.5148,0.5832,0.7675,0.7476
DecisionTreeRegressor,0.3243,0.3932,0.5695,0.4791,0.7135,0.7027,0.2721,0.3659,0.5216,0.5721,0.7590,0.7482
RandomForestRegressor,0.2425,0.3596,0.4924,0.6106,0.7815,0.7501,0.2570,0.3584,0.5070,0.5957,0.7721,0.7629
GradientBoostingRegressor,0.3215,0.4267,0.5670,0.4836,0.7067,0.6726,0.3325,0.4278,0.5766,0.4770,0.7006,0.6920
AdaBoostRegressor,0.5080,0.5787,0.7128,0.1840,0.5107,0.4264,0.4892,0.5678,0.6994,0.2306,0.5538,0.4657
XGBRegressor,0.2394,0.3571,0.4892,0.6156,0.7851,0.7486,0.2509,0.3534,0.5009,0.6054,0.7782,0.7662
ExtraTreesRegressor,0.2434,0.3595,0.4934,0.6090,0.7808,0.7467,0.2583,0.3590,0.5083,0.5937,0.7713,0.7623
LinearRegression,0.4424,0.4889,0.6651,0.2895,0.5391,0.5600,0.4523,0.4904,0.6725,0.2886,0.5377,0.5674
KNeighborsRegressor,0.3155,0.4066,0.5617,0.4933,0.7097,0.6770,0.3068,0.3972,0.5539,0.5175,0.7249,0.7080
SVR,0.3561,0.4254,0.5968,0.4280,0.6654,0.6589,0.3720,0.4256,0.6099,0.4149,0.6531,0.6728


In [53]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-7.033420598698948, -6.882779903564511, -5.78...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.98146378965068, -5.884366103410312, -6.87...","[-6.95097841793298, -5.931880529448997, -6.896...","[0.02711160812063874, 0.13850142935351215, 0.0..."
1,DecisionTreeRegressor,"[-6.96, -7.0, -6.244999999999999, -5.07, -4.52...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.96, -4.68, -7.0, -7.0, -6.24, -7.0, -6.85...","[-6.975999999999999, -5.3759999999999994, -7.0...","[0.019595917942265444, 0.3905688159594928, 0.0..."
2,RandomForestRegressor,"[-6.8793999999999995, -6.8736, -6.084451666666...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9571, -5.758466666666669, -6.920999999999...","[-6.97254, -5.792093333333334, -6.949029999999...","[0.012863063398740342, 0.15250660371858626, 0...."
3,GradientBoostingRegressor,"[-6.670394283331979, -6.573049771858012, -5.55...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.973662767850369, -5.436523678342338, -6.6...","[-7.099387227896604, -5.4561890697232185, -6.7...","[0.1329502965336974, 0.2127746961400684, 0.083..."
4,AdaBoostRegressor,"[-6.009644305482466, -5.631870047543624, -5.57...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.009644305482466, -5.40948412698412, -6.00...","[-6.0247339964926585, -5.565781814399225, -5.9...","[0.07323140366865247, 0.10040587064101908, 0.0..."
5,XGBRegressor,"[-6.907524, -6.831999, -5.7792754, -4.9965405,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.1276627, -5.8423376, -6.872247, -7.077364...","[-7.0776534, -5.754608, -6.963098, -6.675601, ...","[0.058941577, 0.0860931, 0.12133074, 0.5768696..."
6,ExtraTreesRegressor,"[-6.687800000000001, -6.651700000000002, -6.24...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9633, -6.171550000000003, -6.878499999999...","[-6.970020000000001, -6.244430000000002, -6.81...","[0.004929665303041319, 0.10362229296826089, 0...."
7,LinearRegression,"[-5.283034319456934, -5.897349887441942, -5.50...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.014198544459722, -4.647953044030484, -5.7...","[-6.201154640096897, -4.703570186624462, -5.67...","[0.09686307795803216, 0.06640006114278649, 0.0..."
8,KNeighborsRegressor,"[-6.453333333333333, -7.0, -6.496666666666666,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.986666666666667, -5.603333333333333, -7.0...","[-6.9893333333333345, -6.294, -6.824, -5.40799...","[0.005333333333333102, 0.4281142111373762, 0.2..."
9,SVR,"[-6.6885441796836975, -6.896273037930905, -5.4...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.16101096598508, -6.034182574682377, -6.76...","[-7.16662541369198, -6.127234330381708, -6.761...","[0.026675762447656962, 0.08093443051449516, 0...."


In [54]:
df_SubstructureCount_fp.to_csv('Results/Fingerprints/Results_Substructure_Count_fp.csv')
pred_df.to_csv('Results/Fingerprints/Prediction_data_Substructure_Count_fp.csv')


In [ ]:
from sklearn.model_selection import GridSearchCV
import os
import joblib
def train_and_test_predict_with_tuning(models, param_grids, X_train, y_train, X_test, y_test, save_dir):
   
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []
        test_predictions_folds = []

        best_params = None

        # hyperparameter tuning 
        if model_name in param_grids and param_grids[model_name]:
            default_params = model.get_params()
            print(model_name, ': Default params', default_params)
            grid_search = GridSearchCV(
                estimator=model, 
                param_grid=param_grids[model_name], 
                cv=kf,
                scoring='neg_mean_squared_error', 
                n_jobs=-1)
            grid_search.fit(X_train, y_train)
            model = grid_search.best_estimator_
            best_params = grid_search.best_params_
            print(model_name)
            print(": best params",best_params)
        else:
            default_params = model.get_params()
            print(model_name, ': Default params', default_params)
            best_params = {}
            print(model_name, ':Used Default params')

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -3.9)  
            test_predictions_folds.append(predictions_test_fold)

        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)

        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test Predictions folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,
            'Best Parameters': best_params
        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }
        # Save the model
        model_path = os.path.join(save_dir, f"{model_name}.joblib")
        joblib.dump(model, model_path)
        print(f"Saved {model_name} model to {model_path}")

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df


In [ ]:
param_grids = {
        'ExtraTreesRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'max_depth': [None,1,5, 10, 20],
            'min_samples_split': [2, 5, 10]
        },
        'LGBMRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'learning_rate': [0.001, 0.01, 0.05, 0.1],
            'num_leaves': [31, 50, 100]
        },
        'DecisionTreeRegressor': {
            'max_depth': [None, 10, 20, 50, 100],
            'min_samples_split': [2, 5, 10]
        },
        'RandomForestRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'max_depth': [None, 1, 5, 10, 20],
            'min_samples_split': [2, 5, 10]
        },
        'GradientBoostingRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'learning_rate': [0.001, 0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7, 10]
        },
        'AdaBoostRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'learning_rate': [0.001, 0.01, 0.1, 1.0]
        },
        'SVR': {
            'C': [0.001, 0.1, 1, 10],
            'epsilon': [0.1, 0.2, 0.5],
            'gamma': [0.001, 0.1, 1, 10]
        },
        'KNeighborsRegressor': {
            'n_neighbors': [3, 5, 10],
            'weights': ['uniform', 'distance']
        },
        'MLPRegressor': {
            'hidden_layer_sizes': [(50,), (100,), (50, 50)],
            'learning_rate': ['constant', 'adaptive'],
            'max_iter': [100,200, 400]
}
    }


In [ ]:
#All fingerprints const rem Hyperparametric tuning
df_train = pd.read_csv('Fingerprints/Train/All_fingerprints_train.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('Fingerprints/Test/All_fingerprints_test.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
print("X_train shape: ",X_train.shape)
print("X_test shape: ",X_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
save_dir = 'fingerprints_results/Models_All_const_rem_fingerprints_HPT/'
os.makedirs(save_dir, exist_ok=True)
result_df, prediction_df = train_and_test_predict_with_tuning(models, param_grids, X_train,y_train, X_test,  y_test)
result_df

In [ ]:
prediction_df

In [ ]:
result_df.to_csv('fingerprints_results/Results_All_const_rem_fingerprints_HPT.csv')
prediction_df.to_csv('fingerprints_results/Prediction_data_All_const_rem_fingerprints_HPT.csv')